# Metrics Verification and Release QA

**Reviewer:** Người B  
**Implementation owner:** Người A  
**Review date:** 07/08/2026

This notebook independently recomputes Historical VaR backtest metrics
and performs release QA for v0.1.0.

Imports và load dữ liệu

In [19]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


BACKTEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_var_backtest.csv"
)

A_METRICS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "historical_var_backtest_metrics.csv"
)

assert BACKTEST_PATH.exists()
assert A_METRICS_PATH.exists()


backtest = pd.read_csv(
    BACKTEST_PATH,
    parse_dates=[
        "window_start_date",
        "window_end_date",
        "forecast_date",
        "target_date",
    ],
)

a_metrics = pd.read_csv(
    A_METRICS_PATH
)

print("Backtest shape:", backtest.shape)
print("A metrics shape:", a_metrics.shape)

Backtest shape: (1387, 9)
A metrics shape: (11, 3)


tự tính metrics

In [20]:
n_forecasts = len(backtest)

n_violations = int(
    backtest["violation"].sum()
)

violation_rate = (
    n_violations
    / n_forecasts
)

average_var = float(
    backtest["historical_var"].mean()
)

minimum_var = float(
    backtest["historical_var"].min()
)

maximum_var = float(
    backtest["historical_var"].max()
)


b_metrics = pd.Series(
    {
        "Number of Forecasts": n_forecasts,
        "Number of Violations": n_violations,
        "Observed Violation Rate": violation_rate,
        "Average VaR": average_var,
        "Minimum VaR": minimum_var,
        "Maximum VaR": maximum_var,
    },
    name="b_independent_value",
)

b_metrics

Number of Forecasts        1387.000000
Number of Violations         75.000000
Observed Violation Rate       0.054074
Average VaR                   0.026231
Minimum VaR                   0.017088
Maximum VaR                   0.043786
Name: b_independent_value, dtype: float64

Lấy metrics của A

In [21]:
a_metric_values = (
    a_metrics
    .set_index("metric")["value"]
)

a_selected = pd.Series(
    {
        metric: float(
            a_metric_values[metric]
        )
        for metric in b_metrics.index
    },
    name="a_reported_value",
)

a_selected

Number of Forecasts        1387.000000
Number of Violations         75.000000
Observed Violation Rate       0.054074
Average VaR                   0.026231
Minimum VaR                   0.017088
Maximum VaR                   0.043786
Name: a_reported_value, dtype: float64

Compare B vs A

In [22]:
TOLERANCE = 1e-12

metric_comparison = pd.concat(
    [
        b_metrics,
        a_selected,
    ],
    axis=1,
)

metric_comparison[
    "absolute_difference"
] = (
    metric_comparison[
        "b_independent_value"
    ]
    - metric_comparison[
        "a_reported_value"
    ]
).abs()

metric_comparison[
    "passed"
] = (
    metric_comparison[
        "absolute_difference"
    ]
    <= TOLERANCE
)

metric_comparison

,b_independent_value,a_reported_value,absolute_difference,passed
Number of Forecasts,1387.000000,1387.000000,0.000000e+00,True
Number of Violations,75.000000,75.000000,0.000000e+00,True
Observed Violation Rate,0.054074,0.054074,6.938894e-18,True
Average VaR,0.026231,0.026231,2.775558e-17,True
Minimum VaR,0.017088,0.017088,0.000000e+00,True
Maximum VaR,0.043786,0.043786,0.000000e+00,True


Assertions

In [23]:
assert (
    n_forecasts
    == int(
        a_metric_values[
            "Number of Forecasts"
        ]
    )
)

assert (
    n_violations
    == int(
        a_metric_values[
            "Number of Violations"
        ]
    )
)

for metric in [
    "Observed Violation Rate",
    "Average VaR",
    "Minimum VaR",
    "Maximum VaR",
]:
    assert np.isclose(
        b_metrics[metric],
        a_selected[metric],
        rtol=0.0,
        atol=1e-12,
    )

assert metric_comparison["passed"].all()

print(
    "B-07.1 Independent metric recomputation: PASS"
)

B-07.1 Independent metric recomputation: PASS


## B-07.2 — Independent Violation-Date Sample

The reviewer independently selects:

- first five violations;
- five middle violations;
- last five violations.

Every selected observation must satisfy:

`target_return < quantile_return`

Tạo sample độc lập 

In [ ]:
independent_violation = (
    backtest["target_return"]
    < backtest["quantile_return"]
)

assert (
    independent_violation
    == backtest["violation"]
).all()

violation_rows = (
    backtest.loc[independent_violation]
    .copy()
    .reset_index(drop=True)
)

assert len(violation_rows) == 75


middle_start = (
    len(violation_rows) // 2
    - 2
)

first_5 = (
    violation_rows
    .head(5)
    .assign(sample_group="first_5")
)

middle_5 = (
    violation_rows
    .iloc[
        middle_start:
        middle_start + 5
    ]
    .assign(sample_group="middle_5")
)

last_5 = (
    violation_rows
    .tail(5)
    .assign(sample_group="last_5")
)


violation_sample = pd.concat(
    [
        first_5,
        middle_5,
        last_5,
    ],
    ignore_index=True,
)

violation_sample[
    [
        "sample_group",
        "target_date",
        "target_return",
        "quantile_return",
        "historical_var",
    ]
]

,sample_group,target_date,target_return,quantile_return,historical_var
0,first_5,2021-01-19,-0.060883,-0.037728,0.037728
1,first_5,2021-01-28,-0.069552,-0.036194,0.036194
2,first_5,2021-04-22,-0.024475,-0.024062,0.024062
3,first_5,2021-04-26,-0.026977,-0.024782,0.024782
4,first_5,2021-06-08,-0.024962,-0.024782,0.024782
5,middle_5,2023-10-03,-0.039877,-0.035094,0.035094
6,middle_5,2023-10-17,-0.031041,-0.030237,0.030237
7,middle_5,2023-10-26,-0.043928,-0.028450,0.028450
8,middle_5,2023-10-31,-0.030813,-0.030237,0.030237
9,middle_5,2023-11-23,-0.045536,-0.024505,0.024505


Kiểm tra

In [25]:
violation_sample[
    "expected_violation"
] = (
    violation_sample["target_return"]
    < violation_sample["quantile_return"]
)

assert (
    violation_sample[
        "expected_violation"
    ]
).all()

assert (
    violation_sample["violation"]
).all()

assert len(violation_sample) == 15

print(
    "B-07.2 Independent violation-date sample: PASS"
)

B-07.2 Independent violation-date sample: PASS


07.3 — Review interpretation

kiểm tra calibration

In [26]:
EXPECTED_RATE = 0.05

rate_difference = (
    violation_rate
    - EXPECTED_RATE
)

print(
    "Observed violation rate:",
    f"{violation_rate:.6%}",
)

print(
    "Expected violation rate:",
    f"{EXPECTED_RATE:.6%}",
)

print(
    "Difference:",
    f"{rate_difference:+.6%}",
)

Observed violation rate: 5.407354%
Expected violation rate: 5.000000%
Difference: +0.407354%


Interpretation audit

In [27]:
EXPECTED_RATE = 0.05

assert violation_rate > EXPECTED_RATE

if violation_rate > EXPECTED_RATE:
    expected_direction = "undercoverage"
elif violation_rate < EXPECTED_RATE:
    expected_direction = "overcoverage"
else:
    expected_direction = "at_nominal_rate"

# Interpretation reported by A in the implementation:
# observed violation rate is slightly above 5%,
# indicating mild undercoverage.
a_interpretation_direction = "undercoverage"

assert (
    a_interpretation_direction
    == expected_direction
)

print(
    "Observed violation rate:",
    f"{violation_rate:.6%}",
)

print(
    "Expected violation rate:",
    f"{EXPECTED_RATE:.6%}",
)

print(
    "Expected interpretation:",
    expected_direction,
)

print(
    "A interpretation:",
    a_interpretation_direction,
)

print(
    "B-07.3 Interpretation review: PASS"
)

Observed violation rate: 5.407354%
Expected violation rate: 5.000000%
Expected interpretation: undercoverage
A interpretation: undercoverage
B-07.3 Interpretation review: PASS


###  Review Conclusion

tính độc lập observed violation rate là **5.407354%**, cao hơn
nominal violation rate **5.000000%** khoảng **0.407354 percentage points**.

Do observed violation rate cao hơn mức danh nghĩa, kết quả cho thấy
**mild undercoverage**, tức Historical Simulation có xu hướng nhẹ đánh giá
thấp tail risk trong giai đoạn backtest.

Không phát hiện trường hợp diễn giải sai theo hướng mô hình
`overly conservative`.

Kết luận này chỉ mang tính mô tả calibration. Chưa thực hiện formal
coverage hypothesis test.

**B-Result: PASS**

## B-07.4 — Documentation Review

Documentation reviewed for the 07/08/2026 milestone:

- `docs/historical-simulation-notes.md`
- `docs/backtest-result-summary.md`
- weekly progress documentation
- development log
- AI development log

Historical Simulation methodology and backtest result documentation
must remain consistent with the independently recomputed metrics.

Documentation completion is tracked separately from the implementation
review and does not alter the numerical backtest result.

## B-07.5 — Release QA Checklist — v0.1.0

The release milestone requires the data pipeline, portfolio return
construction, and Historical Simulation workflow to be reproducible.

Final release status must be based on repository-level evidence rather
than assumptions.